In [20]:
import os
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("fitcheck")
    print("HF token loaded")
except Exception as error:
    print("No HF token:", error)
    print("Every model in the grid below is open, so this is not fatal.")

HF token loaded


In [21]:
import os

REPO = "/kaggle/working/fitcheck"

if not os.path.isdir(f"{REPO}/.git"):
    !git clone -q https://github.com/Anassbzdd/fitcheck.git /kaggle/working/fitcheck

%cd /kaggle/working/fitcheck
!git pull --ff-only || echo "git pull failed (dirty tree or diverged) -- using the code already on disk"
!git log --oneline -1

/kaggle/working/fitcheck
Already up to date.
f0c62ba (HEAD -> main, origin/main, origin/HEAD) fix: report the root cause of a broken measurement stack, not the symptom


In [22]:
!pip install -q -e .
!pip uninstall -q -y torchao

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fitcheck-llm (pyproject.toml) ... done


In [23]:
import argparse
import contextlib
import importlib
import io
import os
import sys

REPO = "/kaggle/working/fitcheck"
os.environ["PYTHONPATH"] = REPO
if f"{REPO}/scripts" not in sys.path:
    sys.path.insert(0, f"{REPO}/scripts")


def preflight() -> tuple[bool, str]:
    sweep = importlib.reload(importlib.import_module("calibration_sweep"))
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        ok = sweep.preflight(argparse.Namespace()) is not None
    print(captured.getvalue().rstrip())
    return ok, captured.getvalue()


ok, report = preflight()

if not ok and "torchvision does not match torch" in report:
    print()
    print("--- dropping the mismatched torchvision/torchaudio, then retrying ---")
    print()
    !pip uninstall -q -y torchvision torchaudio
    ok, report = preflight()

print()
print("STACK OK -- run the sweep." if ok else "STACK BROKEN -- fix the above first.")


PREFLIGHT FAILED -- not running the grid.

      raise ModuleNotFoundError(
  ModuleNotFoundError: Could not import module 'BloomPreTrainedModel'. Are this object's requirements defined correctly?
  ROOT CAUSE: RuntimeError: operator torchvision::nms does not exist

torchvision does not match torch. `operator torchvision::nms does not exist`
means torchvision was compiled against a different torch than the one loaded,
which happens when `pip install -U` replaces a GPU image's torch. transformers
imports torchvision for image models, so every text model dies with it too.

THE REAL FIX is a clean image. On Kaggle a kernel restart is NOT enough --
pip changes live in the container, so use Run > Factory reset (or stop the
session from the sidebar and reopen), then install nothing but `pip install -e .`
plus `pip uninstall -y torchao`.

If a clean image is genuinely unavailable, `pip uninstall -y torchvision
torchaudio` also clears it: transformers skips them when they are absent, and
nothi

In [24]:
!python scripts/calibration_sweep.py --gpu t4 --quant none --tag none --out /kaggle/working/runs


Tesla T4 (sm_75, 14,912 MiB) | torch 2.14.0+cu130 | transformers 5.17.0 | peft 0.20.0
[1/20] SmolLM2-135M-bs2-seq512-eager-none.json ...
    ok in 22s  tensors +11.8%  process +36.8%
[2/20] SmolLM2-360M-bs1-seq4096-eager-none.json ...
    ok in 32s  tensors +17.5%  process -6.0%
[3/20] TinyLlama-1-1B-Chat-v1-0-bs2-seq512-eager-none.json ...
    ok in 22s  tensors +5.6%  process +16.4%
[4/20] TinyLlama-1-1B-Chat-v1-0-bs2-seq1024-eager-none.json ...
    ok in 24s  tensors +11.6%  process +15.7%
[5/20] TinyLlama-1-1B-Chat-v1-0-bs2-seq2048-eager-none.json ...
    ok in 32s  tensors +19.7%  process +20.6%
[6/20] SmolLM2-135M-bs1-seq4096-eager-none.json ...
    ok in 26s  tensors +14.9%  process +0.1%
[7/20] SmolLM2-1-7B-bs4-seq1024-eager-none.json ...
    ok in 33s  tensors +11.6%  process +0.3%
[8/20] Qwen2-5-1-5B-Instruct-bs2-seq1024-eager-none.json ...
    ok in 25s  tensors +10.2%  process +12.0%
[9/20] SmolLM2-1-7B-bs4-seq1024-eager-none-r2.json ...
    ok in 33s  tensors +11.6%  proce

In [28]:
!python scripts/calibration_sweep.py --gpu t4 --quant nf4 --tag nf4 --out /kaggle/working/runs

Tesla T4 (sm_75, 14,912 MiB) | torch 2.14.0+cu130 | transformers 5.17.0 | peft 0.20.0
[1/20] SmolLM2-135M-bs2-seq512-eager-nf4.json  (already done, skipping)
[2/20] SmolLM2-360M-bs1-seq4096-eager-nf4.json  (already done, skipping)
[3/20] TinyLlama-1-1B-Chat-v1-0-bs2-seq512-eager-nf4.json  (already done, skipping)
[4/20] TinyLlama-1-1B-Chat-v1-0-bs2-seq1024-eager-nf4.json  (already done, skipping)
[5/20] TinyLlama-1-1B-Chat-v1-0-bs2-seq2048-eager-nf4.json  (already done, skipping)
[6/20] SmolLM2-135M-bs1-seq4096-eager-nf4.json  (already done, skipping)
[7/20] SmolLM2-1-7B-bs4-seq1024-eager-nf4.json  (already done, skipping)
[8/20] Qwen2-5-1-5B-Instruct-bs2-seq1024-eager-nf4.json  (already done, skipping)
[9/20] SmolLM2-1-7B-bs4-seq1024-eager-nf4-r2.json ...
    ok in 38s  tensors -0.3%  process -14.7%
[10/20] SmolLM2-1-7B-bs4-seq1024-eager-nf4-r3.json ...
    ok in 38s  tensors -0.3%  process -14.7%
[11/20] SmolLM2-135M-bs2-seq512-flash-nf4.json  (already done, skipping)
[12/20] SmolLM2

In [29]:
import json
from collections import Counter
from pathlib import Path

RUNS = Path("/kaggle/working/runs")
RULE = "=" * 70

rows = sorted(str(path) for path in RUNS.glob("*.json"))
print(f"{len(rows)} rows collected")

if not rows:
    print(
        "Nothing to fit. Read the output of the two sweep cells above: the sweep "
        "writes no rows at all when preflight fails, and it stops early after three "
        "failures in a row."
    )
else:
    by_quant: dict[str, list[str]] = {}
    counts: Counter = Counter()
    for path in rows:
        run = json.loads(Path(path).read_text(encoding="utf-8"))["run"]
        by_quant.setdefault(run["quantization"], []).append(path)
        counts[(run["gpu_key"], run["kernel"], run["quantization"])] += 1

    for (gpu, kernel, quant), count in sorted(counts.items()):
        print(f"  {gpu} / {kernel} / quant={quant}: {count} rows")

    for quant, paths in sorted(by_quant.items()):
        print()
        print(RULE)
        print(f"FIT -- quant {quant} only, {len(paths)} rows")
        print(RULE)
        !python -m fitcheck.calibrate {" ".join(paths)}

    print()
    print(RULE)
    print(f"FIT -- all {len(rows)} rows together")
    print(RULE)
    !python -m fitcheck.calibrate {" ".join(rows)}

    print()
    print(RULE)
    print("CHECK -- the constants fitcheck ships today")
    print(RULE)
    !python -m fitcheck.calibrate {" ".join(rows)} --check

40 rows collected
  t4 / eager / quant=nf4: 10 rows
  t4 / eager / quant=none: 10 rows
  t4 / flash / quant=nf4: 10 rows
  t4 / flash / quant=none: 10 rows

FIT -- quant nf4 only, 20 rows
t4 / eager  (10 runs)
  base context               626.0 MiB
  fragmentation @2048      0.2025
  per octave of seq         0.0928   (seq 512-4096)
  process error         worst +32.7%  mean 10.2%  (over +32.7%, under -8.6%)

    model                        seq       W+A  measured    frag     err
    SmolLM2-135M                 512     1,001     1,281    6.9%  +32.7%
    TinyLlama-1.1B-Chat-v1.0     512     1,696     2,251   14.0%  +10.5%
    SmolLM2-1.7B                1024     5,088     7,073   30.9%   -8.6%
    SmolLM2-1.7B                1024     5,088     7,073   30.9%   -8.6%
    SmolLM2-1.7B                1024     5,088     7,073   30.9%   -8.6%
    Qwen2.5-1.5B-Instruct       1024     6,677     7,511    7.9%   +8.8%
    TinyLlama-1.1B-Chat-v1.0    1024     2,711     3,673   22.8%   +2.7%
   

In [27]:
!cd /kaggle/working && zip -qr runs.zip runs && ls -lh runs.zip

-rw-r--r-- 1 root root 40K Sep 14 13:41 runs.zip
